# 🏥 PediCare AI — Fine-tuning QLoRA pour la synthèse du journal pédiatrique

## Architecture QLoRA
```
Modèle base : Mistral-7B-Instruct-v0.3 (gratuit sur HuggingFace)
Quantization : 4-bit NF4 (bitsandbytes) → ~4 GB GPU
Adaptateurs  : LoRA rank=8, alpha=16 (via PEFT)
GPU requis   : T4 16GB (Google Colab gratuit ✅)
```

## Pipeline
```
Dataset JSON → Tokenisation → QLoRA Training → Merge → Export → HuggingFace Hub
```

## Avant de commencer
1. **Runtime** → Changer le type de runtime → GPU (T4)
2. **HuggingFace token** → https://huggingface.co/settings/tokens (token READ)
3. Exécuter les cellules dans l'ordre ↓

## 📦 Cellule 1 — Installation des dépendances

In [ ]:
# Vérification GPU disponible
!nvidia-smi

# Installation des bibliothèques QLoRA
!pip install -q \
    transformers==4.44.0 \
    peft==0.12.0 \
    bitsandbytes==0.43.3 \
    trl==0.10.1 \
    datasets==2.21.0 \
    accelerate==0.34.2 \
    scipy \
    sentencepiece \
    huggingface_hub

print('✅ Dépendances installées')

## 🔑 Cellule 2 — Authentification HuggingFace

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Option A : Token stocké dans les secrets Colab (recommandé)
# → Colab : icône 🔑 dans la barre latérale → Ajouter un secret 'HF_TOKEN'
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('✅ Connecté via secret Colab')
except:
    # Option B : Saisie manuelle
    print('⚠️  Secret HF_TOKEN non trouvé — saisie manuelle')
    login()  # Ouvre une invite de saisie

## 📊 Cellule 3 — Chargement et préparation du dataset

In [ ]:
import json
from datasets import Dataset

# ── Option A : Dataset depuis le fichier JSON du projet ──────────────────
# Uploadez qlora-journal-dataset.json via Fichiers → Importer
DATASET_PATH = '/content/qlora-journal-dataset.json'  # à adapter

# ── Option B : Dataset intégré (utilisé si fichier absent) ───────────────
DATASET_INLINE = [
    {
        'instruction': 'Tu es un assistant médical pédiatrique. Synthétise les observations du journal parental en une synthèse structurée utile pour le pédiatre.',
        'input': 'Observations (30 jours) :\n• 01/03 [sommeil] Se réveille 3 fois par nuit\n• 05/03 [santé] Légère toux, pas de fièvre\n• 15/03 [santé] Fièvre 38.2°C pendant 2 jours\n• 28/03 [comportement] Reprend ses jeux normalement\nEnfant : 18 mois, garçon.',
        'output': '**Résumé de la période**\nMois marqué par troubles du sommeil et épisode fébrile bénin résolu.\n\n**Tendances observées**\n- Sommeil : 3 réveils/nuit persistants\n- Santé : fièvre 38.2°C, 2 jours, résolue\n\n**Questions pour le pédiatre**\n1. Les réveils nocturnes sont-ils liés à la poussée dentaire ?\n\n**Points de vigilance**\nSurveiller la persistance des troubles du sommeil au-delà de 2 semaines.'
    }
]

# Chargement
try:
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
    examples = data['examples']
    print(f'✅ Dataset chargé depuis fichier : {len(examples)} exemples')
except FileNotFoundError:
    examples = DATASET_INLINE
    print(f'⚠️  Fichier non trouvé — utilisation du dataset intégré : {len(examples)} exemples')

# ── Formatage au format Alpaca (instruction + input → output) ─────────────
ALPACA_TEMPLATE = """### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n{output}"""

def format_example(example):
    """Formate un exemple au format Alpaca pour l'entraînement."""
    return {
        'text': ALPACA_TEMPLATE.format(
            instruction=example['instruction'],
            input=example['input'],
            output=example['output']
        )
    }

formatted = [format_example(e) for e in examples]
dataset = Dataset.from_list(formatted)

# Split train/validation (80/20)
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f'\n📊 Dataset préparé :')
print(f'   Train      : {len(dataset["train"])} exemples')
print(f'   Validation : {len(dataset["test"])} exemples')
print(f'\n📝 Exemple formaté :')
print(dataset['train'][0]['text'][:500] + '...')

## 🤖 Cellule 4 — Chargement du modèle en 4-bit (QLoRA)

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

# ── Modèle base : Mistral-7B (gratuit, excellent en français) ─────────────
MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'

print(f'📥 Chargement du modèle {MODEL_NAME} en 4-bit...')
print(f'   GPU disponible : {torch.cuda.get_device_name(0)}')
print(f'   VRAM disponible : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Configuration 4-bit NF4 (cœur de QLoRA) ──────────────────────────────
# NF4 = Normal Float 4-bit : quantization optimisée pour les poids
# double_quant = quantize aussi les constantes de quantization → encore moins de RAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Chargement en 4-bit
    bnb_4bit_quant_type='nf4',      # Normal Float 4 (meilleur pour LLM)
    bnb_4bit_compute_dtype=torch.float16,  # Calculs en float16
    bnb_4bit_use_double_quant=True  # Double quantization → économie mémoire
)

# ── Chargement tokenizer ─────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'  # Évite les warnings d'attention

# ── Chargement modèle en 4-bit ───────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',           # Distribution automatique sur GPU
    trust_remote_code=True,
    torch_dtype=torch.float16
)
model.config.use_cache = False                    # Désactive KV cache pendant le training
model.config.pretraining_tp = 1

# Mémoire utilisée
used_mem = torch.cuda.memory_allocated() / 1e9
print(f'\n✅ Modèle chargé en 4-bit')
print(f'   VRAM utilisée : {used_mem:.1f} GB / {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'   Paramètres totaux : {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B')

## 🔧 Cellule 5 — Configuration des adaptateurs LoRA (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ── Préparation du modèle pour l'entraînement k-bit ──────────────────────
model = prepare_model_for_kbit_training(model)

# ── Configuration LoRA ────────────────────────────────────────────────────
# rank (r) = dimension des matrices d'adaptation
#   → Plus élevé = plus de paramètres entraînables = meilleure adaptation
#   → Mais aussi plus de VRAM
#   → r=8 = bon équilibre pour T4 gratuit
#
# lora_alpha = facteur de mise à l'échelle (généralement 2x le rank)
# target_modules = couches d'attention à adapter (spécifiques à Mistral)

lora_config = LoraConfig(
    r=8,                      # Rank : dimension des matrices LoRA
    lora_alpha=16,            # Alpha : facteur de scaling (2 × rank)
    target_modules=[          # Modules d'attention ciblés (Mistral)
        'q_proj',             # Query projection
        'k_proj',             # Key projection
        'v_proj',             # Value projection
        'o_proj',             # Output projection
        'gate_proj',          # MLP gate
        'up_proj',            # MLP up
        'down_proj'           # MLP down
    ],
    lora_dropout=0.05,        # Dropout pour régularisation
    bias='none',              # Pas de bias dans les adaptateurs
    task_type='CAUSAL_LM'     # Modèle de langage causal
)

# Application des adaptateurs LoRA au modèle
model = get_peft_model(model, lora_config)

# Statistiques des paramètres entraînables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print('✅ Adaptateurs LoRA configurés')
print(f'   Paramètres entraînables : {trainable:,} ({100 * trainable / total:.2f}% du total)')
print(f'   Paramètres gelés        : {total - trainable:,}')
print(f'   Ratio d\'efficacité QLoRA : x{total // trainable}')

## 🏋️ Cellule 6 — Entraînement QLoRA

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# ── Configuration de l'entraînement ──────────────────────────────────────
OUTPUT_DIR = '/content/pedicare-qlora-journal'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Epochs et batch
    num_train_epochs=5,           # 5 epochs pour un petit dataset
    per_device_train_batch_size=2, # Batch size (T4 gratuit)
    gradient_accumulation_steps=4, # Gradient accumulé = batch effectif de 8

    # Optimisation mémoire
    gradient_checkpointing=True,   # Réduit la VRAM (recompute les activations)
    fp16=True,                     # Mixed precision float16
    optim='paged_adamw_32bit',     # Optimizer paginé (QLoRA recommandé)

    # Learning rate
    learning_rate=2e-4,            # LR standard pour LoRA
    weight_decay=0.001,
    lr_scheduler_type='cosine',    # Cosine annealing
    warmup_ratio=0.05,             # 5% de warmup

    # Logging et sauvegarde
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    evaluation_strategy='steps',
    eval_steps=20,
    load_best_model_at_end=True,

    # Reproductibilité
    seed=42,
    report_to='none'               # Pas de WandB
)

# ── Trainer SFT (Supervised Fine-Tuning) ─────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    dataset_text_field='text',     # Colonne contenant le texte formaté
    max_seq_length=2048,           # Longueur max des séquences
    packing=False                  # Pas de packing (petits datasets)
)

# ── Lancement de l'entraînement ───────────────────────────────────────────
print('🚀 Démarrage de l\'entraînement QLoRA...')
print(f'   Dataset train : {len(dataset["train"])} exemples')
print(f'   Epochs        : {training_args.num_train_epochs}')
print(f'   Batch effectif : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'   Learning rate : {training_args.learning_rate}')
print()

train_result = trainer.train()

print('\n✅ Entraînement terminé !')
print(f'   Loss finale    : {train_result.training_loss:.4f}')
print(f'   Durée totale   : {train_result.metrics["train_runtime"]:.0f}s')
print(f'   Samples/seconde: {train_result.metrics["train_samples_per_second"]:.2f}')

## 🧪 Cellule 7 — Test du modèle fine-tuné

In [ ]:
from transformers import pipeline

# Test avec un exemple réel du dataset
TEST_INPUT = """Observations (30 jours) :
• 01/03 [sommeil] Se réveille 4 fois par nuit depuis 10 jours
• 04/03 [santé] Nez bouché, légère fièvre 37.8°C
• 07/03 [alimentation] Refuse le biberon du soir depuis 3 jours
• 12/03 [santé] Fièvre disparue, rhume en cours de guérison
• 18/03 [comportement] Très irritable en journée
• 22/03 [développement] Commence à pointer du doigt les objets
• 27/03 [sommeil] Nuits légèrement meilleures, 2 réveils
Enfant : 11 mois, fille. Allergies : aucune."""

INSTRUCTION = 'Tu es un assistant médical pédiatrique. Synthétise les observations du journal parental en une synthèse structurée utile pour le pédiatre.'

# Formatage du prompt
prompt = f"""### Instruction:\n{INSTRUCTION}\n\n### Input:\n{TEST_INPUT}\n\n### Response:\n"""

# Tokenisation
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

# Génération
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.3,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

# Décodage — on n'affiche que la partie générée (après le prompt)
generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('=' * 60)
print('📋 SYNTHÈSE GÉNÉRÉE PAR LE MODÈLE QLoRA :')
print('=' * 60)
print(generated)
print('=' * 60)

## 💾 Cellule 8 — Sauvegarde des adaptateurs LoRA

In [ ]:
import os

ADAPTER_DIR = '/content/pedicare-qlora-adapters'
os.makedirs(ADAPTER_DIR, exist_ok=True)

# Sauvegarde des adaptateurs LoRA UNIQUEMENT (pas le modèle de base)
# Taille : ~40 MB (vs 14 GB pour le modèle complet)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Taille des fichiers sauvegardés
total_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)

print(f'✅ Adaptateurs LoRA sauvegardés dans : {ADAPTER_DIR}')
print(f'   Taille totale : {total_size / 1e6:.1f} MB')
print(f'   Fichiers :')
for f in os.listdir(ADAPTER_DIR):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f'     {f} ({size / 1e6:.1f} MB)')

## 🔀 Cellule 9 — Merge modèle base + adaptateurs (optionnel)

In [ ]:
# OPTIONNEL : Fusion complète modèle base + adaptateurs LoRA
# Résultat : modèle standalone deployable sans PEFT
# Nécessite plus de VRAM (16+ GB) — peut nécessiter Colab Pro

MERGE_DIR = '/content/pedicare-qlora-merged'

try:
    from peft import PeftModel
    import torch

    print('🔀 Fusion modèle base + adaptateurs LoRA...')

    # Recharger le modèle de base en float16 (pas en 4-bit pour le merge)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map='auto'
    )

    # Charger les adaptateurs
    merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

    # Fusion
    merged_model = merged_model.merge_and_unload()

    # Sauvegarde
    merged_model.save_pretrained(MERGE_DIR)
    tokenizer.save_pretrained(MERGE_DIR)

    print(f'✅ Modèle fusionné sauvegardé dans : {MERGE_DIR}')

except torch.cuda.OutOfMemoryError:
    print('⚠️  OOM — Le merge nécessite plus de VRAM')
    print('   → Utilisez les adaptateurs seuls (Cellule 10)')
    print('   → Ou activez Colab Pro pour accéder au GPU A100')

## 🚀 Cellule 10 — Export vers HuggingFace Hub

In [ ]:
# Push des adaptateurs vers HuggingFace Hub
# Permet de charger le modèle depuis le backend Node.js via l'API HuggingFace

HF_USERNAME = 'votre-username'  # ← Remplacer par votre username HuggingFace
REPO_NAME   = 'pedicare-journal-qlora'
REPO_ID     = f'{HF_USERNAME}/{REPO_NAME}'

print(f'📤 Publication vers : https://huggingface.co/{REPO_ID}')

model.push_to_hub(
    REPO_ID,
    private=True,   # Repo privé (données médicales)
    safe_serialization=True
)
tokenizer.push_to_hub(REPO_ID, private=True)

print(f'\n✅ Modèle publié !')
print(f'   URL : https://huggingface.co/{REPO_ID}')
print(f'   Pour utiliser dans Node.js : voir README du projet')

## 📊 Cellule 11 — Évaluation qualitative

In [ ]:
# Évaluation sur plusieurs cas de test
TEST_CASES = [
    {
        'label': 'Nourrisson 6 mois — début diversification',
        'input': 'Observations : \n• 01/01 [alimentation] Début diversification\n• 05/01 [santé] Selles molles\n• 10/01 [développement] Tient assis seul\nEnfant : 6 mois, garçon.'
    },
    {
        'label': 'Enfant 3 ans — otites répétées',
        'input': 'Observations :\n• 01/02 [santé] 2ème otite en 1 mois\n• 05/02 [comportement] Parle fort, TV forte\n• 10/02 [développement] Retard léger langage\nEnfant : 36 mois, fille.'
    },
    {
        'label': 'Bébé 9 mois — éruption cutanée + fièvre',
        'input': 'Observations :\n• 02/03 [santé] Fièvre 39.8°C soudaine\n• 04/03 [santé] Éruption tronc après fièvre\n• 06/03 [santé] Éruption disparue, bien\nEnfant : 9 mois, garçon.'
    }
]

INSTRUCTION = 'Tu es un assistant médical pédiatrique. Synthétise les observations du journal parental en une synthèse structurée utile pour le pédiatre.'

for i, case in enumerate(TEST_CASES, 1):
    print(f'\n{'='*60}')
    print(f'TEST {i} : {case["label"]}')
    print('='*60)

    prompt = f'### Instruction:\n{INSTRUCTION}\n\n### Input:\n{case["input"]}\n\n### Response:\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(generated)

## 📥 Cellule 12 — Télécharger les adaptateurs localement

Pour intégrer dans le backend PediCare (Node.js via API HuggingFace)

In [ ]:
import shutil

# Compresser les adaptateurs pour téléchargement
ZIP_PATH = '/content/pedicare-qlora-adapters.zip'
shutil.make_archive(
    '/content/pedicare-qlora-adapters',
    'zip',
    ADAPTER_DIR
)

import os
zip_size = os.path.getsize(ZIP_PATH) / 1e6
print(f'✅ Archive créée : {ZIP_PATH}')
print(f'   Taille : {zip_size:.1f} MB')
print()
print('📥 Pour télécharger :')
print('   Fichiers (panneau gauche) → Clic droit sur pedicare-qlora-adapters.zip → Télécharger')
print()
print('📌 Ensuite, placez le dossier dans :')
print('   server/models/qlora-journal/')